# Lab 03-1 - DBSCAN Clustering (Solutions)

In [ ]:
## Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sklearn

Examples throughout the lab will use the "moons" and "blobs" toy data sets that were introduced in a previous lab. In both cases $n = 500$ and $p = 2$. Note the colors indicate true cluster information that was used to simulate each dataset.

In [ ]:
## Datasets module
from sklearn import datasets

## Toy Dataset #1 - "moons"
moons = datasets.make_moons(n_samples=500, noise=0.11, random_state=8)
moons_X = moons[0]         # features to use in clustering
moons_labels = moons[1]    # true cluster identities
## Visualization
plt.scatter(x = moons_X[:,0], y = moons_X[:,1], c = moons_labels)
plt.title("Toy Dataset #1 - 'moons'")
plt.show()

## Toy Dataset #2 - "blobs"
blobs = datasets.make_blobs(n_samples=500, cluster_std=2, random_state=8)
blobs_X = blobs[0]         # features to use in clustering
blobs_labels = blobs[1]    # true cluster identities
## Visualization
plt.scatter(x = blobs_X[:,0], y = blobs_X[:,1], c = blobs_labels)
plt.title("Toy Dataset #2 - 'blobs'")
plt.show()

As with $k$-means, DBSCAN relies on distances between points, so we first standardize the features for both the `moons_X` and `blobs_X` datasets.

In [ ]:
from sklearn.preprocessing import StandardScaler

moons_XS = StandardScaler().fit_transform(moons_X)
blobs_XS = StandardScaler().fit_transform(blobs_X)

## Part 1 - DBSCAN Clustering

In this section we'll demonstrate DBSCAN clustering using the "moons" data. The code below fits DBSCAN clusters to these data using the default arguments:

In [ ]:
from sklearn.cluster import DBSCAN

# Instantiate a DBSCAN object and fit it to the moons dataset using the default values 
# of the hyperparameters `eps` = 0.5 and `min_samples` = 5. (We can find these default values in the 
# documentation for the DBSCAN class.)
results_dbscan = DBSCAN().fit(moons_XS)

# The resulting labels can be accessed via the `labels_` attribute of the fitted DBSCAN object. 
# These labels indicate the cluster assignment for each data point, with -1 indicating noise 
# points that do not belong to any cluster.
results_dbscan.labels_

In [ ]:
# We observe that the DBSCAN algorithm has associated a large number of points to the first cluster labeled 0
# Let's confirm with a plot by setting the color of each point to its cluster label.
plt.scatter(x = moons_XS[:,0], y = moons_XS[:,1], c = results_dbscan.labels_)
plt.title("'moons' dataset clustered with DBSCAN default hyperparameters")
plt.show()

We see that the default values of `eps` and `min_samples` are way too large for our data, as they group all of the data into the same cluster. We can try some smaller values:

In [ ]:
# Notice how we set the hyperparameter values inside the DBSCAN constructor.
results_dbscan = DBSCAN(eps=0.1, min_samples=2).fit(moons_XS)
plt.scatter(x = moons_XS[:,0], y = moons_XS[:,1], c = results_dbscan.labels_)
plt.title("'moons' dataset clustered with new hyperparameters")
plt.show()

This time we see that our hyperparameter choices are too small. For a simple data set like "moons" we can manually tune them through trial and error to get some really nice looking results. We notice that we were able to get a reasonable fit with 3 detected outliers:

In [ ]:
results_dbscan = DBSCAN(eps=0.28, min_samples=10).fit(moons_XS)
plt.scatter(x = moons_XS[:,0], y = moons_XS[:,1], c = results_dbscan.labels_)
plt.title("'moons' dataset clustered with eps = 0.28, min_samples = 10")
plt.show()

Let's investigate what the cluster labels are for the 500 points for this combination of `eps` and `min_samples`. We have

- Points associated to clusters either labeled `0` or `1`
- Points identified to be outliers labeled `-1`:

In [ ]:
cluster_labels_dbscan = results_dbscan.labels_
cluster_labels_dbscan

In [ ]:
# Count the number of outliers, then divide by number of points. We see that 0.6% of points are 
# identified as outliers by DBSCAN with these hyperparameter values.
np.sum(cluster_labels_dbscan == -1)/cluster_labels_dbscan.shape[0]

**Question 1**:

The provided starter code includes

1. A function `run_DBSCAN()` that accepts the inputs: `eps`, `min_samples`, `data` and applies DBSCAN clustering to data using the given parameters. The function returns the percentage of data that was deemed to be an outlier for that combination of parameters.
1. A *search_grid* of values of `eps` to consider. It uses looping and the function above to explore a sequence of parameter values to find the percentage of the "moons" data set identified as outliers for each value of `eps`. 

Using this starter code:

- **Part A**: Find values of (`eps`, `min_samples`) such that 1% of values are identified as outliers. In this case, you will hold `min_samples=10`, but search over different values of `eps`. Note that you could also do the opposite: hold `eps` constant, and search over different values of `min_samples`
- **Part B**: Using `min_samples=10` and the value of `eps` you found in Part A, display a scatter plot of the resulting DBSCAN cluster assignments.

In [ ]:
# Starter code of function
def run_DBSCAN(eps, min_samples, data):
    results_dbscan = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(data)
    return np.sum(results_dbscan == -1)/results_dbscan.shape[0]

# Test it by comparing to the previous results: should be 0.6%
run_DBSCAN(eps = 0.28, min_samples = 10, data = moons_XS)

In [ ]:
# Starter code of search grid
search_grid = np.linspace(start = 0.01, stop = 2, num = 10)
print("Values of epsilon to search over:", search_grid, "\n")
prop_outliers = np.zeros(search_grid.shape[0])

for i in range(search_grid.shape[0]):
    prop_outliers[i] = run_DBSCAN(eps = search_grid[i], min_samples = 10, data = moons_XS)
    print(f"eps = {search_grid[i]:.5f} -> {prop_outliers[i]:.1%} outliers")

In [ ]:
# Solution code for Q1.A)
# The given search_grid shows 1% outliers falls between eps = 0.23111 (4.8%) and eps = 0.45222 (0.0%),
# so we zoom in on that interval with a finer grid.
search_grid = np.linspace(start = 0.23111, stop = 0.45222, num = 20)
print("Values of epsilon to search over:", search_grid, "\n")
prop_outliers = np.zeros(search_grid.shape[0])

for i in range(search_grid.shape[0]):
    prop_outliers[i] = run_DBSCAN(eps = search_grid[i], min_samples = 10, data = moons_XS)
    print(f"eps = {search_grid[i]:.5f} -> {prop_outliers[i]:.1%} outliers")

**Solution for Q1.A):** The 5th element of the search grid, `eps ≈ 0.27766`, yields exactly 1% outliers (5 of the 500 points) when `min_samples = 10`. Note several `eps` values can give the same percentage.

In [ ]:
# Solution code for Q1.B)
best_eps = 0.27766
results_dbscan_1pct = DBSCAN(eps = best_eps, min_samples = 10).fit(moons_XS)
results_dbscan_1pct_labels = results_dbscan_1pct.labels_

plt.scatter(x = moons_XS[:,0], y = moons_XS[:,1], c = results_dbscan_1pct_labels)
plt.title(f"DBSCAN with eps = {best_eps:.4f}, min_samples = 10 ({np.mean(results_dbscan_1pct_labels == -1):.1%} outliers)")
plt.show()

## Part 2 - Application

**Question 2**:

Your task in this portion of the lab is to apply your knowledge of clustering to analyze data that describe the workforce characteristics of college degree holders in a variety of fields. Let's start with some exploratory data analysis (EDA). 


In [ ]:
jobs = pd.read_csv("../data/majors.csv")

# Look at first 10 rows of the raw data
jobs.head(n = 10)

In [ ]:
# Additionally we can look at variable types, the amount of missing values (null count), n = 34
jobs.info()

Your goal is to prepare a brief report that groups the workforce in different degree fields into clusters based upon commonalities, such as similar salaries, similar racial/ethnic/gender distributions, similar rates of graduate degree attainment, etc. You should explore at least two of the following clustering algorithms:

1. $k$-means clustering
1. agglomerative clustering
1. DBSCAN clustering

You should be careful to standardize the data if necessary and only cluster using variables you deem relevant to the analysis. You must:

1. Explain and justify all of your choices in a short "methods" paragraph, using graphs or numerical information as appropriate
1. Describe your final cluster assignments in a "results" paragraph, again using graphs or numerical information as appropriate.

This question is intentionally open-ended and is aimed at preparing you to use clustering methods on your own in the future. I encourage you to check-in with me if you ever feel unsure about what you've done.

**Solution for Q2):**

**Methods.** The data set has $n = 34$ majors and no missing values. I used only a subset of the variables covering gender, race/ethnicity, degree attainment, employment, and median income. These are measured on very different scales (percentages from 0-100 vs. incomes near \$100,000), so I standardized all of them; without standardizing, the two income variables would dominate every distance calculation.

In [ ]:
# Solution code for Q2)

# Keep only these variables:
features = ['Per_Female', 'Per_White', 'Per_Black', 'Per_Asian', 'Per_Hispanic',
            'Per_Bachelors', 'Per_Masters', 'Per_Professional_Degree', 'Per_PhD',
            'Per_Employed', 'Per_Unemployed', 'Bach_Med_Income', 'Grad_Med_Income']

# Run a standard scaler to standardize the features. This is important because percentages 
# (0-100) and incomes (~100,000) are on wildly different scales.
# 
# Note we run .fit_transform():
# - instead of .fit() followed by .transform() because we want to both fit the scaler 
# to the data and transform the data in one step.
# - on jobs[features] because we are dropping Major, Category, and Workforce
jobs_XS = StandardScaler().fit_transform(jobs[features])
jobs_XS.shape

### 1. k-Means

Let's run k-Means clustering. Let's compute the elbow plot first to identify a good value of k to use

In [ ]:
from sklearn.cluster import KMeans

k_grid = range(2, 8)
inertias = []
for k in k_grid:
    km = KMeans(n_clusters=k, random_state=76, n_init=10).fit(jobs_XS)
    inertias.append(km.inertia_)

plt.plot(k_grid, inertias, marker='o')
plt.xlabel('k')
plt.ylabel('Objective')
plt.title('Elbow plot for KMeans clustering')
plt.show()

Using k = 4 seems to be a reasonable choice

In [ ]:
# Fit KMeans with k = 4
kmeans_labels = KMeans(n_clusters=4, random_state=76, n_init=10).fit_predict(jobs_XS)

print("KMeans labels:", kmeans_labels)

In [ ]:
# This seems to be the engineering + finance cluster
jobs[kmeans_labels == 0][['Major'] + features]

In [ ]:
# This seems to be the social services cluster, very high female representation
jobs[kmeans_labels == 1][['Major'] + features]

In [ ]:
# These seem to have high PhD rates
jobs[kmeans_labels == 2][['Major'] + features]

In [ ]:
# A real mix of majors
jobs[kmeans_labels == 3][['Major'] + features]

### 2. Agglomerative Clustering

Let's run Agglomerative clustering with Ward linkage. We'll re-use the value of k from earlier

In [ ]:
from sklearn.cluster import AgglomerativeClustering

ward_labels = AgglomerativeClustering(n_clusters=4, linkage='ward').fit_predict(jobs_XS)
print("Ward labels:", ward_labels)

In [ ]:
# This is similar to the social services cluster we saw with KMeans
jobs[ward_labels == 0][['Major'] + features]

In [ ]:
# This seems to be the engineering + finance cluster we saw with KMeans
jobs[ward_labels == 1][['Major'] + features]

In [ ]:
# This seems to be a mix?
jobs[ward_labels == 2][['Major'] + features]

In [ ]:
# Only 3 majors in this one?
jobs[ward_labels == 3][['Major'] + features]

### 3. DBSCAN

Let's run DBSCAN and try different values of epsilon for min_samples = 3

In [ ]:
# Sweep eps with min_samples = 3
for eps in [1.5, 2.0, 2.5, 3.0, 3.5]:
    lab = DBSCAN(eps=eps, min_samples=3).fit_predict(jobs_XS)
    print(f"eps = {eps}: {len(set(lab) - {-1})} cluster(s), {np.sum(lab == -1)}/{len(jobs)} outliers")

These don't look useful, since we either have a very large number of outliers, or mostly 0 or 1 clusters. 

In [ ]:
# Sweep eps with min_samples = 5
for eps in [1.5, 2.0, 2.5, 3.0, 3.5]:
    lab = DBSCAN(eps=eps, min_samples=5).fit_predict(jobs_XS)
    print(f"eps = {eps}: {len(set(lab) - {-1})} cluster(s), {np.sum(lab == -1)}/{len(jobs)} outliers")

Again, we don't seem to have a useful number of clusters or a reasonable number of outliers

### 4. Putting it all together

The $k$-means solution at k = 4 is interpretable, and it is mostly about two
things: who majors in a field, and how much credentialing that field requires.

- Cluster 0 (n = 7) is the high-earning, male-majority technical block:
computer science, the engineering disciplines, economics, and finance
- Cluster 1 (n = 6) is almost its mirror image: education,
social work, and psychology, which are female dominated
- Cluster 2 (n = 7) could be a graduate school cluster; mathematics,
biology, chemistry, political science, English, history. A low share of the workforce stops 
at a bachelor's, with elevated rates of PhDs and professional degrees.
- Cluster 3 is the leftover bulk (n = 14), mostly business and the applied
fields.  As a contrast to Cluster 2, these could be majors where a
bachelor's degree is the terminal credential.